In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

print(PROJECT_DIR)
print(list(PROJECT_DIR.iterdir()))

/content/drive/MyDrive/hinglish-next-word-predictor
[PosixPath('/content/drive/MyDrive/hinglish-next-word-predictor/notebooks'), PosixPath('/content/drive/MyDrive/hinglish-next-word-predictor/data')]


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/hinglish-next-word-predictor")
RAW_DIR = PROJECT_DIR / "data" / "raw"

ALL_TXT = RAW_DIR / "all.txt"

with open(ALL_TXT, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
print("\nFirst 50 lines:\n")

for i, line in enumerate(lines[:50], start=1):
    print(f"{i:03}: {repr(line)}")

Total lines: 1399949

First 50 lines:

001: 'sanatan\tHI\n'
002: '0809\tHI\n'
003: 'tiding\tEN\n'
004: 'luna\tEN\n'
005: 'in\tEN\n'
006: 'ham\tHI\n'
007: 'bowling\tEN\n'
008: 'krte\tHI\n'
009: 'hai\tHI\n'
010: 'aap\tHI\n'
011: 'apne\tHI\n'
012: 'stump\tEN\n'
013: 'bhjao\tHI\n'
014: '\n'
015: 'aap\tHI\n'
016: 'ne\tHI\n'
017: 'kaha\tHI\n'
018: 'tha\tHI\n'
019: 'apne\tHI\n'
020: 'video\tEN\n'
021: 'main\tHI\n'
022: 'ki\tHI\n'
023: 'koi\tHI\n'
024: 'video\tEN\n'
025: 'share\tEN\n'
026: 'kare\tHI\n'
027: 'him\tEN\n'
028: 'toh\tHI\n'
029: 'yeh\tHI\n'
030: 'link\tEN\n'
031: 'hai\tHI\n'
032: 'aap\tHI\n'
033: 'aksar\tHI\n'
034: 'poochti\tHI\n'
035: 'hai\tHI\n'
036: 'maulana\tHI\n'
037: 'saab\tHI\n'
038: 'se\tHI\n'
039: 'ki\tHI\n'
040: 'aurat\tHI\n'
041: 'ko\tHI\n'
042: 'jannat\tHI\n'
043: 'main\tHI\n'
044: 'kya\tHI\n'
045: 'mile\tHI\n'
046: 'ga\tHI\n'
047: 'ismain\tHI\n'
048: 'maulana\tHI\n'
049: 'ne\tHI\n'
050: 'yah\tHI\n'


In [ ]:
blank_lines = 0

for line in lines:
    if not line.strip():
        blank_lines += 1

print("Blank lines:", blank_lines)

Blank lines: 44452


In [ ]:
print("First 20 non-empty lines:")

count = 0
for line in lines:
    if line.strip():
        print(repr(line))
        count += 1

    if count == 20:
        break


First 20 non-empty lines:
'sanatan\tHI\n'
'0809\tHI\n'
'tiding\tEN\n'
'luna\tEN\n'
'in\tEN\n'
'ham\tHI\n'
'bowling\tEN\n'
'krte\tHI\n'
'hai\tHI\n'
'aap\tHI\n'
'apne\tHI\n'
'stump\tEN\n'
'bhjao\tHI\n'
'aap\tHI\n'
'ne\tHI\n'
'kaha\tHI\n'
'tha\tHI\n'
'apne\tHI\n'
'video\tEN\n'
'main\tHI\n'


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
parser_code = r'''
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

INPUT_FILE = PROJECT_DIR / "data" / "raw" / "all.txt"
OUTPUT_FILE = PROJECT_DIR / "data" / "processed" / "sentences.txt"
TAGGED_OUTPUT_FILE = PROJECT_DIR / "data" / "processed" / "sentences_with_tags.txt"


def parse_all_txt():
    sentences = []
    tagged_sentences = []

    current_tokens = []
    current_tags = []

    with INPUT_FILE.open("r", encoding="utf-8") as f:
        for line_number, raw_line in enumerate(f, start=1):
            line = raw_line.strip()

            # Blank line = end of sentence
            if not line:
                if current_tokens:
                    sentences.append(" ".join(current_tokens))
                    tagged_sentences.append(
                        " ".join(
                            f"{token}/{tag}"
                            for token, tag in zip(
                                current_tokens, current_tags
                            )
                        )
                    )

                    current_tokens = []
                    current_tags = []

                continue

            parts = line.split("\t")

            # Ignore malformed lines safely
            if len(parts) != 2:
                print(
                    f"Warning: malformed line {line_number}: "
                    f"{raw_line!r}"
                )
                continue

            token, tag = parts

            token = token.strip()
            tag = tag.strip()

            if not token:
                continue

            current_tokens.append(token)
            current_tags.append(tag)

    # Handle a final sentence without a trailing blank line
    if current_tokens:
        sentences.append(" ".join(current_tokens))
        tagged_sentences.append(
            " ".join(
                f"{token}/{tag}"
                for token, tag in zip(current_tokens, current_tags)
            )
        )

    OUTPUT_FILE.write_text(
        "\n".join(sentences),
        encoding="utf-8"
    )

    TAGGED_OUTPUT_FILE.write_text(
        "\n".join(tagged_sentences),
        encoding="utf-8"
    )

    print(f"Sentences parsed: {len(sentences)}")
    print(f"Saved: {OUTPUT_FILE}")
    print(f"Saved: {TAGGED_OUTPUT_FILE}")


if __name__ == "__main__":
    parse_all_txt()
'''

parser_path = PROJECT_DIR / "preprocessing" / "parse_all.py"
parser_path.write_text(parser_code, encoding="utf-8")

print(f"Created: {parser_path}")

Created: /content/drive/MyDrive/hinglish-next-word-predictor/preprocessing/parse_all.py


In [ ]:
!python "/content/drive/MyDrive/hinglish-next-word-predictor/preprocessing/parse_all.py"


Sentences parsed: 44453
Saved: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/sentences.txt
Saved: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/sentences_with_tags.txt


In [ ]:
sentences_file = PROJECT_DIR / "data" / "processed" / "sentences.txt"

with sentences_file.open("r", encoding="utf-8") as f:
    sentences = f.readlines()

print("Number of reconstructed sentences:", len(sentences))

print("\nFirst 5 sentences:\n")

for i, sentence in enumerate(sentences[:5], start=1):
    print(f"{i}: {sentence.strip()}")

Number of reconstructed sentences: 44453

First 5 sentences:

1: sanatan 0809 tiding luna in ham bowling krte hai aap apne stump bhjao
2: aap ne kaha tha apne video main ki koi video share kare him toh yeh link hai aap aksar poochti hai maulana saab se ki aurat ko jannat main kya mile ga ismain maulana ne yah bataya hai thodi funny bhi hai beech beech main
3: gill main first time bb13 ko apne friends ke yaha dekhi mujhe bigboss me koi intrest nahi tha but jaise hi shehnaaz ki entry huwi us din se bb13 ke har ek episodes dekhi but aaj bhi main bigboss house me shehnaaz ko hi dhoondhti hoo i miss you shehnaaz and miss you bb13 shehnaazgiii
4: ek nari ko duniya ke samne apne nagan sarir ko pradarshan karna sobha nahi deta aisi isthiti kewal band kamre me apne pati ke sath hona chahiye apke pass srif 3 din hai ye sab rok do hari om namo narayan thanks
5: jatti7 ki ho gea sadi stubborn jatti sad it doesn t look nyc eh tn hasdi wadia lgdi apne gang nal


In [ ]:
tagged_file = PROJECT_DIR / "data" / "processed" / "sentences_with_tags.txt"

with tagged_file.open("r", encoding="utf-8") as f:
    tagged_sentences = f.readlines()

print("\nFirst 3 tagged sentences:\n")

for sentence in tagged_sentences[:3]:
    print(sentence.strip())


First 3 tagged sentences:

sanatan/HI 0809/HI tiding/EN luna/EN in/EN ham/HI bowling/EN krte/HI hai/HI aap/HI apne/HI stump/EN bhjao/HI
aap/HI ne/HI kaha/HI tha/HI apne/HI video/EN main/HI ki/HI koi/HI video/EN share/EN kare/HI him/EN toh/HI yeh/HI link/EN hai/HI aap/HI aksar/HI poochti/HI hai/HI maulana/HI saab/HI se/HI ki/HI aurat/HI ko/HI jannat/HI main/HI kya/HI mile/HI ga/HI ismain/HI maulana/HI ne/HI yah/HI bataya/HI hai/HI thodi/HI funny/EN bhi/HI hai/HI beech/EN beech/EN main/HI
gill/EN main/HI first/EN time/EN bb13/HI ko/HI apne/HI friends/EN ke/HI yaha/HI dekhi/HI mujhe/HI bigboss/HI me/HI koi/HI intrest/EN nahi/HI tha/HI but/EN jaise/HI hi/HI shehnaaz/HI ki/HI entry/EN huwi/HI us/HI din/HI se/HI bb13/HI ke/HI har/HI ek/HI episodes/EN dekhi/HI but/EN aaj/HI bhi/HI main/HI bigboss/HI house/EN me/HI shehnaaz/HI ko/HI hi/HI dhoondhti/HI hoo/HI i/EN miss/EN you/EN shehnaaz/HI and/EN miss/EN you/EN bb13/HI shehnaazgiii/HI


In [ ]:
from pathlib import Path
from collections import Counter
import statistics
import re

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

SENTENCES_FILE = PROJECT_DIR / "data" / "processed" / "sentences.txt"
TAGGED_FILE = PROJECT_DIR / "data" / "processed" / "sentences_with_tags.txt"


# -----------------------------
# Load reconstructed sentences
# -----------------------------

with SENTENCES_FILE.open("r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]

# Tokenize by whitespace
tokenized = [sentence.split() for sentence in sentences]

sentence_lengths = [len(tokens) for tokens in tokenized]

# Flatten all tokens
all_tokens = [token for tokens in tokenized for token in tokens]

token_counts = Counter(all_tokens)


# -----------------------------
# Basic statistics
# -----------------------------

print("========== DATASET SUMMARY ==========\n")

print("Number of sentences:", len(sentences))
print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(token_counts))

print("\nSentence length statistics:")
print("Minimum:", min(sentence_lengths))
print("Maximum:", max(sentence_lengths))
print("Mean:", round(statistics.mean(sentence_lengths), 2))
print("Median:", statistics.median(sentence_lengths))

print("\nTop 30 most frequent tokens:")
for token, count in token_counts.most_common(30):
    print(f"{token:20} {count}")


# -----------------------------
# Duplicate analysis
# -----------------------------

sentence_counts = Counter(sentences)

duplicate_sentences = {
    sentence: count
    for sentence, count in sentence_counts.items()
    if count > 1
}

print("\n========== DUPLICATES ==========\n")

print("Unique sentences:", len(sentence_counts))
print("Duplicate sentence types:", len(duplicate_sentences))

if duplicate_sentences:
    print("\nExample duplicates:")
    for sentence, count in list(duplicate_sentences.items())[:10]:
        print(f"{count}x -> {sentence}")


# -----------------------------
# Very short / very long
# -----------------------------

print("\n========== LENGTH CHECK ==========\n")

short_sentences = [
    sentence for sentence, tokens in zip(sentences, tokenized)
    if len(tokens) <= 2
]

long_sentences = [
    sentence for sentence, tokens in zip(sentences, tokenized)
    if len(tokens) >= 100
]

print("Sentences with <= 2 tokens:", len(short_sentences))
print("Sentences with >= 100 tokens:", len(long_sentences))

print("\nExamples of short sentences:")
for sentence in short_sentences[:10]:
    print("-", sentence)

print("\nExamples of long sentences:")
for sentence in long_sentences[:5]:
    print("-", sentence[:300], "...")


# -----------------------------
# Suspicious token analysis
# -----------------------------

def looks_suspicious(token):
    # Keep normal alphabetic, numeric, mixed alphanumeric,
    # apostrophe and common punctuation forms.
    return not re.fullmatch(r"[A-Za-z0-9_'.!?,-]+", token)


suspicious_tokens = Counter(
    token for token in all_tokens
    if looks_suspicious(token)
)

print("\n========== SUSPICIOUS TOKENS ==========\n")
print("Number of suspicious unique tokens:", len(suspicious_tokens))

for token, count in suspicious_tokens.most_common(50):
    print(f"{token:30} {count}")


========== DATASET SUMMARY ==========

Number of sentences: 44453
Total tokens: 1355497
Unique tokens: 75910

Sentence length statistics:
Minimum: 7
Maximum: 77
Mean: 30.49
Median: 28

Top 30 most frequent tokens:
apne                 47620
hai                  32066
ko                   22932
ki                   19106
ke                   17749
se                   17356
to                   15996
ka                   13354
me                   12139
bhi                  11580
h                    11250
ho                   10306
k                    10154
aur                  10001
hi                   9304
nahi                 8852
nhi                  7494
ye                   7244
kar                  6507
aap                  6320
kya                  6189
or                   5992
na                   5947
liye                 5936
jo                   5525
hain                 5063
ne                   5060
pe                   4679
koi                  4623
ek                

In [ ]:
uppercase_tokens = [
    token
    for token in all_tokens
    if any(c.isupper() for c in token)
]

print("Tokens containing uppercase characters:", len(uppercase_tokens))

if uppercase_tokens:
    print("\nExamples:")
    print(uppercase_tokens[:50])


Tokens containing uppercase characters: 0


In [ ]:
from pathlib import Path
import random

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

INPUT_FILE = PROJECT_DIR / "data" / "processed" / "sentences.txt"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"


# Reproducible random seed
SEED = 42

# Split ratios
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10


def split_dataset():
    # Load sentences
    with INPUT_FILE.open("r", encoding="utf-8") as f:
        sentences = [line.strip() for line in f if line.strip()]

    print("Total sentences:", len(sentences))

    # Reproducible shuffle
    rng = random.Random(SEED)
    rng.shuffle(sentences)

    total = len(sentences)

    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)

    train_sentences = sentences[:train_end]
    val_sentences = sentences[train_end:val_end]
    test_sentences = sentences[val_end:]

    # Save
    TRAIN_FILE.write_text(
        "\n".join(train_sentences),
        encoding="utf-8"
    )

    VAL_FILE.write_text(
        "\n".join(val_sentences),
        encoding="utf-8"
    )

    TEST_FILE.write_text(
        "\n".join(test_sentences),
        encoding="utf-8"
    )

    print("\nSplit complete:")
    print("Train:", len(train_sentences))
    print("Validation:", len(val_sentences))
    print("Test:", len(test_sentences))

    print("\nSaved:")
    print(TRAIN_FILE)
    print(VAL_FILE)
    print(TEST_FILE)


split_dataset()

Total sentences: 44453

Split complete:
Train: 35562
Validation: 4445
Test: 4446

Saved:
/content/drive/MyDrive/hinglish-next-word-predictor/data/processed/train.txt
/content/drive/MyDrive/hinglish-next-word-predictor/data/processed/validation.txt
/content/drive/MyDrive/hinglish-next-word-predictor/data/processed/test.txt


In [ ]:
for file in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    with file.open("r", encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())

    print(f"{file.name}: {count} sentences")


train.txt: 35562 sentences
validation.txt: 4445 sentences
test.txt: 4446 sentences


In [ ]:
for file in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    print(f"\n===== {file.name} =====")

    with file.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(line.strip())


===== train.txt =====
ja hi rhi ha toh coups ko apne pocket mein chupa kar le ana aur uss 6ft bangchan ko bhi
petrol diesel par 400 tax badakar apne jeben bhar rahe hain aur sath mein apne doston ki aur kehte hain ki free de rahe hain
jise tumne islam laane ke baad kiya hai kiunke maine jannat me apne aage tumhaare juton ki chaap aawaz suni hai bilaal razi allah anhu ne arz kiya ke maine to apne nazdeek isse zyada ummeed ka koi kaam nahi kiya ke jab maine raat ya din me kisi waqt bhi wuzu kiya to mai 2

===== validation.txt =====
shukla abe nalle kutte tu pahle apne us nalle janwar asim ko sambhal jiski suar jaisi shakal hai use per to thukne ka bhi man nahin karta
muharram faqat libaas ko tabdeel krna nhn balke apne kirdaar ko bhi tabdeel krne ka mahina hai
youre the one being a home wrecker meri husband ki picture ke saath shaadi karke apne aap ko pati nahi kya samajna lag rahi ho jyp jitna tum mention karahi ho tumhara hi lagraha hai it makes sense too cuz both of you are delulus



In [ ]:
def load_set(file):
    with file.open("r", encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())


train_set = load_set(TRAIN_FILE)
val_set = load_set(VAL_FILE)
test_set = load_set(TEST_FILE)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

Train ∩ Validation: 71
Train ∩ Test: 67
Validation ∩ Test: 15


In [ ]:
from pathlib import Path
import random

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

INPUT_FILE = PROJECT_DIR / "data" / "processed" / "sentences.txt"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

SEED = 42


def split_unique_sentences():
    # Load original reconstructed sentences
    with INPUT_FILE.open("r", encoding="utf-8") as f:
        sentences = [line.strip() for line in f if line.strip()]

    print("Original sentence occurrences:", len(sentences))

    # Remove exact duplicate sentence strings
    unique_sentences = list(dict.fromkeys(sentences))

    print("Unique sentences:", len(unique_sentences))
    print("Removed duplicates:", len(sentences) - len(unique_sentences))

    # Reproducible shuffle
    rng = random.Random(SEED)
    rng.shuffle(unique_sentences)

    total = len(unique_sentences)

    train_end = int(total * 0.80)
    val_end = train_end + int(total * 0.10)

    train_sentences = unique_sentences[:train_end]
    val_sentences = unique_sentences[train_end:val_end]
    test_sentences = unique_sentences[val_end:]

    # Save
    TRAIN_FILE.write_text(
        "\n".join(train_sentences),
        encoding="utf-8"
    )

    VAL_FILE.write_text(
        "\n".join(val_sentences),
        encoding="utf-8"
    )

    TEST_FILE.write_text(
        "\n".join(test_sentences),
        encoding="utf-8"
    )

    print("\nNew split:")
    print("Train:", len(train_sentences))
    print("Validation:", len(val_sentences))
    print("Test:", len(test_sentences))


split_unique_sentences()

Original sentence occurrences: 44453
Unique sentences: 43922
Removed duplicates: 531

New split:
Train: 35137
Validation: 4392
Test: 4393


In [ ]:
def load_set(file):
    with file.open("r", encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())


train_set = load_set(TRAIN_FILE)
val_set = load_set(VAL_FILE)
test_set = load_set(TEST_FILE)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [ ]:
from collections import defaultdict, Counter
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"


def load_sentences(path):
    with path.open("r", encoding="utf-8") as f:
        return [line.strip().split() for line in f if line.strip()]


train_sentences = load_sentences(TRAIN_FILE)
val_sentences = load_sentences(VAL_FILE)
test_sentences = load_sentences(TEST_FILE)

print("Train sentences:", len(train_sentences))
print("Validation sentences:", len(val_sentences))
print("Test sentences:", len(test_sentences))


Train sentences: 35137
Validation sentences: 4392
Test sentences: 4393


In [ ]:
ngram_counts = defaultdict(Counter)

for sentence in train_sentences:
    if len(sentence) < 5:
        continue

    for i in range(4, len(sentence)):
        context = tuple(sentence[i-4:i])
        target = sentence[i]

        ngram_counts[context][target] += 1

print("Number of unique 4-word contexts:", len(ngram_counts))

Number of unique 4-word contexts: 904634


In [ ]:
def predict_ngram(context_words, top_k=5):
    context_words = context_words[-4:]
    context = tuple(context_words)

    candidates = ngram_counts.get(context)

    if not candidates:
        return []

    return candidates.most_common(top_k)

In [ ]:
examples = [
    ["main", "kal", "office", "ja"],
    ["mujhe", "kal", "subah", "se"],
    ["aap", "kya", "kar", "rahe"],
]

for example in examples:
    print("\nContext:", " ".join(example))
    print("Predictions:", predict_ngram(example))


Context: main kal office ja
Predictions: []

Context: mujhe kal subah se
Predictions: []

Context: aap kya kar rahe
Predictions: [('he', 1)]


In [ ]:
from collections import defaultdict, Counter

# Build n-gram counts for 1-, 2-, 3-, and 4-word contexts
ngram_models = {
    1: defaultdict(Counter),
    2: defaultdict(Counter),
    3: defaultdict(Counter),
    4: defaultdict(Counter),
}

overall_counts = Counter()

for sentence in train_sentences:
    for i, target in enumerate(sentence):
        overall_counts[target] += 1

        for n in range(1, 5):
            if i >= n:
                context = tuple(sentence[i-n:i])
                ngram_models[n][context][target] += 1


def predict_backoff(context_words, top_k=5):
    context_words = context_words[-4:]

    # Try longest available context first
    for n in range(min(4, len(context_words)), 0, -1):
        context = tuple(context_words[-n:])

        candidates = ngram_models[n].get(context)

        if candidates:
            return candidates.most_common(top_k)

    # Final fallback: globally most frequent words
    return overall_counts.most_common(top_k)

In [ ]:
examples = [
    ["main", "kal", "office", "ja"],
    ["mujhe", "kal", "subah", "se"],
    ["aap", "kya", "kar", "rahe"],
]

for example in examples:
    print("\nContext:", " ".join(example))
    print("Predictions:", predict_backoff(example))


Context: main kal office ja
Predictions: [('rhi', 1)]

Context: mujhe kal subah se
Predictions: [('lagabhag', 1), ('trend', 1), ('mein', 1), ('raat', 1), ('isliye', 1)]

Context: aap kya kar rahe
Predictions: [('he', 1)]


In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

model.to(device)
model.eval()

print("Model loaded successfully.")

Device: cpu


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )


CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)

model.to(device)
model.eval()

print("Model loaded successfully.")

Device: cuda


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [ ]:
import torch

def top_next_tokens(text, top_k=10):
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits for the last input position
    next_token_logits = outputs.logits[:, -1, :]

    # Convert logits to probabilities
    probs = torch.softmax(next_token_logits, dim=-1)

    top_probs, top_ids = torch.topk(probs, top_k)

    results = []

    for prob, token_id in zip(
        top_probs[0].tolist(),
        top_ids[0].tolist()
    ):
        token = tokenizer.decode(
            [token_id],
            skip_special_tokens=True
        )

        results.append((token, prob))

    return results

In [ ]:
test_prompts = [
    "aap kya kar rahe",
    "main kal office",
    "mujhe kal",
    "aap ne kaha tha",
    "tum kya karte",
]

for prompt in test_prompts:
    print("\n" + "=" * 60)
    print("PROMPT:", prompt)

    predictions = top_next_tokens(prompt, top_k=10)

    for token, probability in predictions:
        print(f"{token!r:20} {probability:.4f}")


PROMPT: aap kya kar rahe
' hai'               0.3208
' j'                 0.1539
' h'                 0.0884
' y'                 0.0344
' s'                 0.0278
' he'                0.0278
' ya'                0.0231
' a'                 0.0173
' ja'                0.0155
' ga'                0.0135

PROMPT: main kal office
'\n\n'               0.0874
'\n'                 0.0433
' in'                0.0379
','                  0.0356
' '                  0.0337
':'                  0.0255
' -'                 0.0182
' is'                0.0163
'.'                  0.0137
' building'          0.0122

PROMPT: mujhe kal
'b'                  0.0935
'te'                 0.0537
'ah'                 0.0529
' hot'               0.0497
'ay'                 0.0467
' b'                 0.0449
' pa'                0.0328
'ori'                0.0326
'u'                  0.0270
'aha'                0.0242

PROMPT: aap ne kaha tha
'?'                  0.2079
'?\n\n'              0.0649
','      

In [ ]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.3 MB/s eta 0:00:00


In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

CUDA: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

OUTPUT_DIR = MODELS_DIR / "qwen3-0.6b-hinglish-lora"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Training data:", TRAIN_FILE)
print("Output:", OUTPUT_DIR)

Project: /content/drive/MyDrive/hinglish-next-word-predictor
Training data: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/train.txt
Output: /content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora


In [ ]:
!pip uninstall -y pyarrow datasets fsspec -q
!pip install -q \
    "pyarrow>=16,<19" \
    "datasets>=3.0,<5" \
    "fsspec>=2024.6.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.12.0 requires datasets>=4.7.0, but you have datasets 4.0.0 which is incompatible.


In [ ]:
!pip install -q -U \
    "datasets>=4.7.0" \
    "pyarrow>=18,<24" \
    "fsspec>=2024.12.0" \
    "transformers>=4.56" \
    "accelerate>=1.9" \
    "peft>=0.17" \
    "trl==1.12.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.6.0 which is incompatible.


In [ ]:
!pip install -q --force-reinstall \
    "fsspec==2025.3.0" \
    "gcsfs==2025.3.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.5/341.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.6/250.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.4/234.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.5/180.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import sys
import torch
import datasets
import pyarrow
import transformers
import peft
import trl
import fsspec

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Datasets:", datasets.__version__)
print("PyArrow:", pyarrow.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("fsspec:", fsspec.__version__)

print("\nCUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
Datasets: 5.0.1
PyArrow: 23.0.1
Transformers: 5.16.1
PEFT: 0.20.0
TRL: 1.12.0
fsspec: 2025.3.0

CUDA: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

dataset = load_dataset(
    "text",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
        "test": str(TEST_FILE),
    }
)

print(dataset)

FileNotFoundError: Unable to find '/content/drive/MyDrive/hinglish-next-word-predictor/data/processed/train.txt'

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Processed directory exists:", PROCESSED_DIR.exists())

if PROCESSED_DIR.exists():
    for file in PROCESSED_DIR.iterdir():
        print(file.name)

Processed directory exists: False


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Processed exists:", PROCESSED_DIR.exists())

Raw directory: /content/drive/MyDrive/hinglish-next-word-predictor/data/raw
Processed directory: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed
Processed exists: True


In [ ]:
INPUT_FILE = RAW_DIR / "all.txt"
SENTENCES_FILE = PROCESSED_DIR / "sentences.txt"
TAGGED_FILE = PROCESSED_DIR / "sentences_with_tags.txt"

sentences = []
tagged_sentences = []

current_tokens = []
current_tags = []

with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line_number, raw_line in enumerate(f, start=1):
        line = raw_line.strip()

        # Blank line = sentence boundary
        if not line:
            if current_tokens:
                sentences.append(" ".join(current_tokens))

                tagged_sentences.append(
                    " ".join(
                        f"{token}/{tag}"
                        for token, tag in zip(current_tokens, current_tags)
                    )
                )

                current_tokens = []
                current_tags = []

            continue

        parts = line.split("\t")

        if len(parts) != 2:
            print(f"Skipping malformed line {line_number}: {raw_line!r}")
            continue

        token, tag = parts

        token = token.strip()
        tag = tag.strip()

        if token:
            current_tokens.append(token)
            current_tags.append(tag)

# Handle final sentence
if current_tokens:
    sentences.append(" ".join(current_tokens))

    tagged_sentences.append(
        " ".join(
            f"{token}/{tag}"
            for token, tag in zip(current_tokens, current_tags)
        )
    )

SENTENCES_FILE.write_text(
    "\n".join(sentences),
    encoding="utf-8"
)

TAGGED_FILE.write_text(
    "\n".join(tagged_sentences),
    encoding="utf-8"
)

print("Reconstructed sentences:", len(sentences))
print("Saved:", SENTENCES_FILE)
print("Saved:", TAGGED_FILE)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/hinglish-next-word-predictor/data/raw/all.txt'

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

print("Drive exists:", DRIVE_ROOT.exists())

PROJECT_DIR = DRIVE_ROOT / "hinglish-next-word-predictor"

print("Project exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("\nProject contents:")
    for path in PROJECT_DIR.iterdir():
        print(" ", path)

RAW_DIR = PROJECT_DIR / "data" / "raw"

print("\nRaw folder exists:", RAW_DIR.exists())

if RAW_DIR.exists():
    print("\nRaw files:")
    for path in RAW_DIR.iterdir():
        print(" ", path.name, "-", round(path.stat().st_size / 1024**2, 2), "MB")

Drive exists: True
Project exists: True

Project contents:
  /content/drive/MyDrive/hinglish-next-word-predictor/models
  /content/drive/MyDrive/hinglish-next-word-predictor/data

Raw folder exists: False


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

DATA_DIR = PROJECT_DIR / "data"

print("Data exists:", DATA_DIR.exists())
print("Data is directory:", DATA_DIR.is_dir())

print("\nContents of data/:")
for path in DATA_DIR.iterdir():
    print(path, " | directory:", path.is_dir(), " | file:", path.is_file())

Data exists: True
Data is directory: True

Contents of data/:
/content/drive/MyDrive/hinglish-next-word-predictor/data/processed  | directory: True  | file: False


In [ ]:
from google.colab import drive

drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [ ]:
drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [ ]:
import os

print("Exists:", os.path.exists("/content/drive"))

if os.path.exists("/content/drive"):
    print("Contents:", os.listdir("/content/drive"))

Exists: True
Contents: ['MyDrive']


In [ ]:
!rm -rf /content/drive

In [ ]:
import os

print("Drive path exists:", os.path.exists("/content/drive"))

Drive path exists: False


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

DATA_DIR = PROJECT_DIR / "data"

print("Project exists:", PROJECT_DIR.exists())
print("\nProject contents:")

for path in PROJECT_DIR.iterdir():
    print(" ", path)

Project exists: True

Project contents:
  /content/drive/MyDrive/hinglish-next-word-predictor/notebooks
  /content/drive/MyDrive/hinglish-next-word-predictor/data
  /content/drive/MyDrive/hinglish-next-word-predictor/preprocessing
  /content/drive/MyDrive/hinglish-next-word-predictor/training
  /content/drive/MyDrive/hinglish-next-word-predictor/models
  /content/drive/MyDrive/hinglish-next-word-predictor/inference
  /content/drive/MyDrive/hinglish-next-word-predictor/evaluation


In [ ]:
RAW_DIR = DATA_DIR / "raw"

print("\nRaw exists:", RAW_DIR.exists())

if RAW_DIR.exists():
    for path in RAW_DIR.iterdir():
        print(
            path.name,
            "-",
            round(path.stat().st_size / (1024 * 1024), 2),
            "MB"
        )


Raw exists: True
hinglishNorm.json - 2.83 MB
all.txt - 10.39 MB


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

ALL_FILE = RAW_DIR / "all.txt"

# Create processed directory
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", ALL_FILE)
print("Exists:", ALL_FILE.exists())
print("Output directory:", PROCESSED_DIR)

Input: /content/drive/MyDrive/hinglish-next-word-predictor/data/raw/all.txt
Exists: True
Output directory: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed


In [ ]:
SENTENCES_FILE = PROCESSED_DIR / "sentences.txt"
TAGGED_FILE = PROCESSED_DIR / "sentences_with_tags.txt"

sentences = []
tagged_sentences = []

current_tokens = []
current_tags = []

with ALL_FILE.open("r", encoding="utf-8") as f:
    for line_number, raw_line in enumerate(f, start=1):

        line = raw_line.strip()

        # Blank line = sentence boundary
        if not line:
            if current_tokens:
                sentences.append(" ".join(current_tokens))

                tagged_sentences.append(
                    " ".join(
                        f"{token}/{tag}"
                        for token, tag in zip(current_tokens, current_tags)
                    )
                )

                current_tokens = []
                current_tags = []

            continue

        parts = line.split("\t")

        if len(parts) != 2:
            print(
                f"Warning: malformed line {line_number}: {raw_line!r}"
            )
            continue

        token, tag = parts

        token = token.strip()
        tag = tag.strip()

        if token:
            current_tokens.append(token)
            current_tags.append(tag)

# Handle final sentence
if current_tokens:
    sentences.append(" ".join(current_tokens))

    tagged_sentences.append(
        " ".join(
            f"{token}/{tag}"
            for token, tag in zip(current_tokens, current_tags)
        )
    )


SENTENCES_FILE.write_text(
    "\n".join(sentences),
    encoding="utf-8"
)

TAGGED_FILE.write_text(
    "\n".join(tagged_sentences),
    encoding="utf-8"
)

print("Reconstructed sentences:", len(sentences))
print("Saved:", SENTENCES_FILE)
print("Saved:", TAGGED_FILE)

Reconstructed sentences: 44453
Saved: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/sentences.txt
Saved: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/sentences_with_tags.txt


In [ ]:
import random

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

# Remove exact duplicates
unique_sentences = list(dict.fromkeys(sentences))

print("Original sentences:", len(sentences))
print("Unique sentences:", len(unique_sentences))
print("Exact duplicates removed:", len(sentences) - len(unique_sentences))

# Reproducible shuffle
rng = random.Random(42)
rng.shuffle(unique_sentences)

total = len(unique_sentences)

train_end = int(total * 0.80)
val_end = train_end + int(total * 0.10)

train_sentences = unique_sentences[:train_end]
val_sentences = unique_sentences[train_end:val_end]
test_sentences = unique_sentences[val_end:]

TRAIN_FILE.write_text(
    "\n".join(train_sentences),
    encoding="utf-8"
)

VAL_FILE.write_text(
    "\n".join(val_sentences),
    encoding="utf-8"
)

TEST_FILE.write_text(
    "\n".join(test_sentences),
    encoding="utf-8"
)

print("\nSplit:")
print("Train:", len(train_sentences))
print("Validation:", len(val_sentences))
print("Test:", len(test_sentences))

Original sentences: 44453
Unique sentences: 43922
Exact duplicates removed: 531

Split:
Train: 35137
Validation: 4392
Test: 4393


In [ ]:
for file in [
    SENTENCES_FILE,
    TAGGED_FILE,
    TRAIN_FILE,
    VAL_FILE,
    TEST_FILE
]:
    print(
        file.name,
        "| exists:", file.exists(),
        "| size:",
        round(file.stat().st_size / (1024 * 1024), 2),
        "MB"
    )

sentences.txt | exists: True | size: 6.47 MB
sentences_with_tags.txt | exists: True | size: 10.35 MB
train.txt | exists: True | size: 5.1 MB
validation.txt | exists: True | size: 0.64 MB
test.txt | exists: True | size: 0.63 MB


In [ ]:
def load_set(path):
    with path.open("r", encoding="utf-8") as f:
        return set(
            line.strip()
            for line in f
            if line.strip()
        )

train_set = load_set(TRAIN_FILE)
val_set = load_set(VAL_FILE)
test_set = load_set(TEST_FILE)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [ ]:
from pathlib import Path
from datasets import load_dataset

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

print("Train exists:", TRAIN_FILE.exists())
print("Validation exists:", VAL_FILE.exists())
print("Test exists:", TEST_FILE.exists())

Train exists: True
Validation exists: True
Test exists: True


In [ ]:
dataset = load_dataset(
    "text",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
        "test": str(TEST_FILE),
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 35137
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 4392
    })
    test: Dataset({
        features: ['text'],
        num_rows: 4393
    })
})


In [ ]:
print("Train example:", dataset["train"][0])
print("Validation example:", dataset["validation"][0])
print("Test example:", dataset["test"][0])

Train example: {'text': 'lo ji col kejariwal khud dekh lo apne maliko ke area ka hall delhi li don ban gayi'}
Validation example: {'text': 'meri mehfil mein karke andhere apni mehfil saja hue hain apne haathon se khanjar chalakar kitna maasoom chehra banaakar inki maasoom nazron ne nasir log paagal banaaye hue hain three dead men i d like to meet nusrat khushwant and jeff buckley'}
Test example: {'text': 'hahahaha bhai tu apne maan me khush hota reh jaise vivek hua tha usne bhi yahi socha tha sir ko finish kar diya but jise rab rakhe usko koun chakhkhe tu bhi yahi hain hum bhi yahi dekh lenge 10 saal bad bhi'}


In [ ]:
import numpy as np
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sample training sentences
sample_size = min(5000, len(dataset["train"]))

sample_texts = [
    dataset["train"][i]["text"]
    for i in range(sample_size)
]

lengths = []

for text in sample_texts:
    token_ids = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    lengths.append(len(token_ids))

print("Sample sentences:", len(lengths))
print("Average Qwen tokens:", round(np.mean(lengths), 2))
print("Median Qwen tokens:", int(np.median(lengths)))
print("Maximum Qwen tokens:", max(lengths))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("99th percentile:", int(np.percentile(lengths, 99)))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Sample sentences: 5000
Average Qwen tokens: 49.65
Median Qwen tokens: 45
Maximum Qwen tokens: 109
95th percentile: 93
99th percentile: 100


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
)

model = model.to("cuda")
model.train()

print("Model device:", next(model.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model device: cuda:0


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

PAD token: <|endoftext|>
PAD token ID: 151643


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'q_proj', 'k_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [ ]:
small_train = dataset["train"].select(
    range(min(32, len(dataset["train"])))
)

small_val = dataset["validation"].select(
    range(min(16, len(dataset["validation"])))
)

print("Small train:", len(small_train))
print("Small validation:", len(small_val))

Small train: 32
Small validation: 16


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

OUTPUT_DIR = MODELS_DIR / "qwen3-0.6b-hinglish-lora"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Train:", TRAIN_FILE)
print("Validation:", VAL_FILE)
print("Test:", TEST_FILE)
print("Output:", OUTPUT_DIR)

Project: /content/drive/MyDrive/hinglish-next-word-predictor
Train: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/train.txt
Validation: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/validation.txt
Test: /content/drive/MyDrive/hinglish-next-word-predictor/data/processed/test.txt
Output: /content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora


In [ ]:
print("Train exists:", TRAIN_FILE.exists())
print("Validation exists:", VAL_FILE.exists())
print("Test exists:", TEST_FILE.exists())

Train exists: True
Validation exists: True
Test exists: True


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "text",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
        "test": str(TEST_FILE),
    }
)

small_train = dataset["train"].select(
    range(min(32, len(dataset["train"])))
)

small_val = dataset["validation"].select(
    range(min(16, len(dataset["validation"])))
)

print(dataset)
print("Small train:", len(small_train))
print("Small validation:", len(small_val))

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 35137
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 4392
    })
    test: Dataset({
        features: ['text'],
        num_rows: 4393
    })
})
Small train: 32
Small validation: 16


In [ ]:
from trl import SFTConfig

dry_run_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "dry_run"),

    num_train_epochs=1,
    learning_rate=1e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,

    max_length=128,

    logging_steps=1,

    eval_strategy="steps",
    eval_steps=4,

    save_strategy="no",

    fp16=True,
    report_to="none",
    seed=42,
)

print("SFTConfig created successfully.")

SFTConfig created successfully.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
).to("cuda")

model.config.pad_token_id = tokenizer.pad_token_id

print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model device: cuda:0


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print("LoRA configured.")

LoRA configured.


In [ ]:
!pip install -q -U "torchao>=0.16.0"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

OUTPUT_DIR = MODELS_DIR / "qwen3-0.6b-hinglish-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch
import torchao
import peft
import trl
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

print("\nCUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
TorchAO: 0.18.0
PEFT: 0.20.0
TRL: 1.12.0
Transformers: 5.16.1
Datasets: 5.0.1

CUDA: True
GPU: Tesla T4


In [ ]:
from trl import SFTTrainer

print("SFTTrainer import: OK")

SFTTrainer import: OK


In [ ]:
from peft import LoraConfig

print("LoraConfig import: OK")

LoraConfig import: OK


In [ ]:
from pathlib import Path
from datasets import load_dataset

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

dataset = load_dataset(
    "text",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
        "test": str(TEST_FILE),
    }
)

small_train = dataset["train"].select(range(32))
small_val = dataset["validation"].select(range(16))

print(dataset)
print("Small train:", len(small_train))
print("Small validation:", len(small_val))

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 35137
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 4392
    })
    test: Dataset({
        features: ['text'],
        num_rows: 4393
    })
})
Small train: 32
Small validation: 16


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
)

model = model.to("cuda")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model device: cuda:0


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print("LoRA configured.")

LoRA configured.


In [ ]:
import inspect
from trl import SFTConfig

print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_com

In [ ]:
print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None =

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print("LoRA configuration created.")

LoRA configuration created.


In [ ]:
from trl import SFTConfig

dry_run_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "dry_run"),

    # Training
    num_train_epochs=1,
    learning_rate=1e-4,

    # T4-safe
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,

    # Your corpus is short
    max_length=128,

    # Evaluation
    eval_strategy="steps",
    eval_steps=4,

    # No checkpoint saving for dry run
    save_strategy="no",

    # Logging
    logging_steps=1,
    report_to="none",

    # GPU
    fp16=True,

    # Reproducibility
    seed=42,

    # Dataset
    dataset_text_field="text",
    packing=False,
)

print("Dry-run configuration created.")

Dry-run configuration created.


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=dry_run_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Trainer created successfully.")

Adding EOS to train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Trainer created successfully.


In [ ]:
trainable_params = sum(
    p.numel()
    for p in trainer.model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in trainer.model.parameters()
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

Total parameters:     600,637,440
Trainable parameters: 4,587,520
Trainable percentage: 0.7638%


In [ ]:
import torch

torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU allocated: 2.24 GB
GPU reserved: 2.26 GB


In [ ]:
dry_run_result = trainer.train()

print("Dry run completed successfully.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
4,6.283797,5.668882,5.420401,468.000000,0.144473
8,5.053201,5.616966,5.397990,922.000000,0.150985
12,5.394837,5.595943,5.394245,1289.000000,0.153437
16,4.940454,5.589158,5.393419,1719.000000,0.153921


Dry run completed successfully.


In [ ]:
print(dry_run_result)

TrainOutput(global_step=16, training_loss=5.604255944490433, metrics={'train_runtime': 20.0232, 'train_samples_per_second': 1.598, 'train_steps_per_second': 0.799, 'total_flos': 6104374050816.0, 'train_loss': 5.604255944490433, 'epoch': 1.0})


In [ ]:
print(
    "GPU allocated after training:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved after training:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU allocated after training: 2.29 GB
GPU reserved after training: 3.32 GB


In [ ]:
import torch

def predict_next_tokens(text, top_k=10):
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = trainer.model(**inputs)

    logits = outputs.logits[:, -1, :]
    probabilities = torch.softmax(logits, dim=-1)

    top_probs, top_ids = torch.topk(
        probabilities,
        k=top_k
    )

    results = []

    for prob, token_id in zip(
        top_probs[0].tolist(),
        top_ids[0].tolist()
    ):
        token = tokenizer.decode(
            [token_id],
            skip_special_tokens=True
        )

        results.append({
            "token": token,
            "probability": prob
        })

    return results

In [ ]:
test_prompts = [
    "aap kya kar rahe",
    "main kal office",
    "mujhe kal",
    "aap ne kaha tha",
    "tum kya karte",
]

for prompt in test_prompts:
    print("\n" + "=" * 60)
    print("PROMPT:", prompt)

    predictions = predict_next_tokens(
        prompt,
        top_k=10
    )

    for item in predictions:
        print(
            f"{item['token']!r:15} "
            f"{item['probability']:.4f}"
        )


PROMPT: aap kya kar rahe
' h'            0.3629
' hai'          0.3356
' ho'           0.0469
' ha'           0.0211
' hi'           0.0211
' he'           0.0162
' a'            0.0075
' j'            0.0073
' ya'           0.0066
' y'            0.0052

PROMPT: main kal office
'\n'            0.0511
'\n\n'          0.0286
' '             0.0278
' is'           0.0205
','             0.0151
' k'            0.0125
' and'          0.0096
' in'           0.0093
' job'          0.0088
' jobs'         0.0086

PROMPT: mujhe kal
'pan'           0.0591
' b'            0.0542
'b'             0.0406
'pa'            0.0309
' k'            0.0266
' pa'           0.0254
' n'            0.0235
' p'            0.0146
' se'           0.0133
' pat'          0.0130

PROMPT: aap ne kaha tha
' k'            0.0781
' ki'           0.0689
' a'            0.0279
' ap'           0.0258
' to'           0.0258
' aur'          0.0207
' na'           0.0198
' us'           0.0186
' '             0.0183
' tha'  

In [ ]:
DRY_RUN_DIR = OUTPUT_DIR / "dry_run"

DRY_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

trainer.save_model(str(DRY_RUN_DIR))
tokenizer.save_pretrained(str(DRY_RUN_DIR))

print("Dry-run adapter saved to:")
print(DRY_RUN_DIR)

Dry-run adapter saved to:
/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora/dry_run


In [ ]:
for path in sorted(DRY_RUN_DIR.iterdir()):
    print(path.name)

README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json
training_args.bin


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
ADAPTER_DIR = str(OUTPUT_DIR / "dry_run")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
).to("cuda")

base_tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR
)

loaded_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

loaded_model.eval()

print("Adapter loaded successfully.")
print("Model device:", next(loaded_model.parameters()).device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Adapter loaded successfully.
Model device: cuda:0


In [ ]:
def quick_predict(text, top_k=10):
    inputs = base_tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    values, ids = torch.topk(probs, top_k)

    results = []

    for value, token_id in zip(values[0], ids[0]):
        token = base_tokenizer.decode(
            [token_id],
            skip_special_tokens=True
        )

        results.append(
            (token, float(value))
        )

    return results


print(
    quick_predict("aap kya kar rahe")
)

[(' h', 0.3623046875), (' hai', 0.3349609375), (' ho', 0.046051025390625), (' ha', 0.021087646484375), (' hi', 0.021087646484375), (' he', 0.01617431640625), (' a', 0.007518768310546875), (' j', 0.0074005126953125), (' ya', 0.006740570068359375), (' y', 0.0052490234375)]


In [ ]:
# Start full Hinglish fine-tuning
train_result = trainer.train()

print("\n========================================")
print("TRAINING COMPLETED")
print("========================================")

print("\nTraining metrics:")
print(train_result.metrics)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
4,5.912747,5.529940,5.343030,2187.000000,0.156525
8,4.771811,5.494152,5.296051,2641.000000,0.155334
12,5.182489,5.476147,5.255235,3008.000000,0.157463
16,4.741035,5.469300,5.256554,3438.000000,0.154075



TRAINING COMPLETED

Training metrics:
{'train_runtime': 29.5662, 'train_samples_per_second': 1.082, 'train_steps_per_second': 0.541, 'total_flos': 6104374050816.0, 'train_loss': 5.283087939023972, 'epoch': 1.0}


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

REAL_OUTPUT_DIR = (
    PROJECT_DIR
    / "models"
    / "qwen3-0.6b-hinglish-lora"
)

REAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:", REAL_OUTPUT_DIR)

Output directory: /content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora


In [ ]:
print("Trainer exists:", "trainer" in globals())

Trainer exists: True


In [ ]:
trainer.save_model(str(REAL_OUTPUT_DIR))
tokenizer.save_pretrained(str(REAL_OUTPUT_DIR))

print("Model saved successfully.")

Model saved successfully.


In [ ]:
for path in sorted(REAL_OUTPUT_DIR.iterdir()):
    print(path.name)

README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
dry_run
tokenizer.json
tokenizer_config.json
training_args.bin


In [ ]:
adapter_file = REAL_OUTPUT_DIR / "adapter_model.safetensors"

if adapter_file.exists():
    print(
        "Adapter size:",
        round(adapter_file.stat().st_size / (1024**2), 2),
        "MB"
    )
else:
    print("Adapter file not found.")

Adapter size: 17.53 MB


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

PROJECT_DIR = "/content/drive/MyDrive/hinglish-next-word-predictor"

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
ADAPTER_DIR = f"{PROJECT_DIR}/models/qwen3-0.6b-hinglish-lora"

# Load base model
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
).to("cuda")

# Load trained LoRA adapter
trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

trained_model.eval()

print("Fine-tuned model loaded.")
print("Device:", next(trained_model.parameters()).device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Fine-tuned model loaded.
Device: cuda:0


In [ ]:
import torch

def predict_tokens(text, top_k=10):
    inputs = base_tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = trained_model(**inputs)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    values, ids = torch.topk(probs, top_k)

    results = []

    for value, token_id in zip(values[0], ids[0]):
        token = base_tokenizer.decode(
            [token_id],
            skip_special_tokens=True
        )

        results.append(
            (token, float(value))
        )

    return results

In [ ]:
test_prompts = [
    "aap kya kar rahe",
    "main kal office",
    "mujhe kal",
    "aap ne kaha tha",
    "tum kya karte",
]

for prompt in test_prompts:
    print("\n" + "=" * 60)
    print("PROMPT:", prompt)

    for token, prob in predict_tokens(prompt, top_k=10):
        print(f"{token!r:15} {prob:.4f}")


PROMPT: aap kya kar rahe
' h'            0.5034
' hai'          0.2268
' ho'           0.0620
' ha'           0.0235
' hi'           0.0154
' he'           0.0094
' hy'           0.0070
' hue'          0.0065
' hu'           0.0059
' y'            0.0056

PROMPT: main kal office
' k'            0.0247
'\n'            0.0233
' '             0.0218
' ka'           0.0192
'\n\n'          0.0123
' job'          0.0123
' jobs'         0.0114
' is'           0.0111
' ke'           0.0093
','             0.0089

PROMPT: mujhe kal
'pan'           0.0625
' b'            0.0526
' pa'           0.0316
'b'             0.0302
' k'            0.0279
'pa'            0.0264
' p'            0.0201
' n'            0.0195
' pat'          0.0183
' pe'           0.0171

PROMPT: aap ne kaha tha
' k'            0.0757
' ki'           0.0489
' ap'           0.0432
' aur'          0.0326
' a'            0.0279
' us'           0.0246
' to'           0.0235
' '             0.0186
' na'           0.0186
' tha'  

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

PROJECT_DIR = "/content/drive/MyDrive/hinglish-next-word-predictor"

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
ADAPTER_DIR = (
    f"{PROJECT_DIR}/models/qwen3-0.6b-hinglish-lora"
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
).to("cuda")

trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

trained_model.eval()

print("Model loaded:", next(trained_model.parameters()).device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model loaded: cuda:0


In [ ]:
from pathlib import Path

TEST_FILE = (
    Path(PROJECT_DIR)
    / "data"
    / "processed"
    / "test.txt"
)

with TEST_FILE.open("r", encoding="utf-8") as f:
    test_sentences = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Test sentences:", len(test_sentences))

Test sentences: 4393


In [ ]:
import torch

def get_word_candidates(
    context,
    top_k=5,
    num_beams=8,
    max_new_tokens=8
):
    inputs = tokenizer(
        context,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            num_return_sequences=num_beams,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    candidates = []

    for output in outputs:
        generated_ids = output[input_length:]

        text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        text = text.strip()

        if not text:
            continue

        # First whitespace-delimited word
        first_word = text.split()[0]

        if first_word:
            candidates.append(first_word)

    # Remove duplicates while keeping order
    unique_candidates = []

    for word in candidates:
        if word not in unique_candidates:
            unique_candidates.append(word)

    return unique_candidates[:top_k]

In [ ]:
examples = [
    "aap kya kar rahe",
    "tum kya karte",
    "mujhe kal",
    "aap ne kaha tha",
    "main kal office",
]

for context in examples:
    candidates = get_word_candidates(
        context,
        top_k=5
    )

    print("\nContext:", context)
    print("Candidates:", candidates)


Context: aap kya kar rahe
Candidates: ['hain']

Context: tum kya karte
Candidates: ['hain']

Context: mujhe kal
Candidates: ['bhi', 'pana', 'nahi']

Context: aap ne kaha tha
Candidates: ['aap', 'kya', 'kahin']

Context: main kal office
Candidates: ['2022-23', '2021-22', '2020-21', '2023-24', '2019-10']


In [ ]:
import random
import torch

random.seed(42)

sample_indices = random.sample(
    range(len(test_sentences)),
    10
)

for idx in sample_indices:
    sentence = test_sentences[idx]
    words = sentence.split()

    context = " ".join(words[:-1])
    actual = words[-1]

    inputs = tokenizer(
        context,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = trained_model(**inputs)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    values, ids = torch.topk(probs, 10)

    print("\n" + "=" * 70)
    print("Context :", context)
    print("Actual  :", actual)
    print("Top raw tokens:")

    for value, token_id in zip(values[0], ids[0]):
        token = tokenizer.decode(
            [token_id],
            skip_special_tokens=True
        )

        print(
            f"  {token!r:15} "
            f"{float(value):.4f}"
        )


Context : is se kaam nhi chalega inko facilities do jese india apne athletes ko deta hai thankyou for giving us hope se kaam nhi chalega ab you re a champion agar tum kisi or nation k hote to gold tmhara hi tha
Actual  : arshadnadeem
Top raw tokens:
  ' aur'          0.0815
  ' to'           0.0687
  ' ap'           0.0560
  ' k'            0.0518
  ''              0.0416
  ' then'         0.0286
  ' tum'          0.0277
  ' a'            0.0191
  ' us'           0.0173
  ' and'          0.0156

Context : bharat ne ball possession asani se khoya hai uss wajah se argentina ka goal hua sirf ball ko apne pass rakh sakte to acha chance ban sakta hai hope 2nd half me hum lead
Actual  : kare
Top raw tokens:
  ' h'            0.1140
  ' k'            0.0930
  ' kar'          0.0530
  ' ho'           0.0327
  ' se'           0.0250
  ' ke'           0.0183
  ' ban'          0.0183
  ' ko'           0.0169
  ' n'            0.0164
  ' hai'          0.0149

Context : jinnah garden phase 1 and 2

In [ ]:
eval_results = trainer.evaluate()

print(eval_results)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
4.741035,5.469300,16,5.256554,3438.000000,0.154075


{'eval_loss': 5.469300270080566, 'eval_entropy': 5.256554484367371, 'eval_num_tokens': 3438.0, 'eval_mean_token_accuracy': 0.15407539904117584}


In [ ]:
import math

if "eval_loss" in eval_results:
    print(
        "Validation perplexity:",
        math.exp(eval_results["eval_loss"])
    )

Validation perplexity: 237.2940928789231


In [ ]:
# ============================================================
# HINGLISH NEXT-WORD PREDICTOR
# End-to-end training + evaluation
# Qwen3-0.6B-Base + LoRA
# Google Colab T4
# ============================================================

import os
import gc
import math
import json
import time
import random
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/hinglish-next-word-predictor"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "qwen3-0.6b-hinglish-lora-final"

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

TRAIN_FILE = PROCESSED_DIR / "train.txt"
VAL_FILE = PROCESSED_DIR / "validation.txt"
TEST_FILE = PROCESSED_DIR / "test.txt"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. LOAD DATA
# ============================================================

dataset = load_dataset(
    "text",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
        "test": str(TEST_FILE),
    },
)

print("\nDataset:")
print(dataset)


# ============================================================
# 3. TOKENIZER + BASE MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)

model.config.pad_token_id = tokenizer.pad_token_id

if DEVICE == "cuda":
    model = model.to("cuda")

print("\nBase model loaded.")


# ============================================================
# 4. LoRA
# ============================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)


# ============================================================
# 5. TRAINING CONFIGURATION
# ============================================================

training_args = SFTConfig(
    output_dir=str(MODEL_DIR),

    # Training
    num_train_epochs=2,
    learning_rate=1e-4,

    # T4
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Corpus statistics support this
    max_length=128,

    # Memory
    gradient_checkpointing=True,
    fp16=True,

    # Evaluation / checkpoints
    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # Logging
    logging_strategy="steps",
    logging_steps=25,

    # Reproducibility
    seed=SEED,

    # Dataset
    dataset_text_field="text",
    packing=True,

    # Keep output clean
    report_to="none",
)


# ============================================================
# 6. TRAIN
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("\nStarting training...\n")

start_train = time.time()

train_result = trainer.train()

train_time = time.time() - start_train

print("\nTraining finished.")
print("Training time:",
      round(train_time / 60, 2),
      "minutes")


# ============================================================
# 7. SAVE MODEL
# ============================================================

trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print("\nModel saved to:")
print(MODEL_DIR)


# ============================================================
# 8. VALIDATION
# ============================================================

eval_results = trainer.evaluate()

print("\nValidation results:")
print(eval_results)

if "eval_loss" in eval_results:
    try:
        ppl = math.exp(eval_results["eval_loss"])
    except OverflowError:
        ppl = float("inf")

    print("Validation perplexity:", ppl)


# ============================================================
# 9. CLEANUP TRAINER MODEL FROM MEMORY
# ============================================================

del trainer
del model

gc.collect()
torch.cuda.empty_cache()

print("\nGPU memory cleared.")


# ============================================================
# 10. LOAD FINAL ADAPTER FOR INFERENCE
# ============================================================

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
).to("cuda")

trained_model = PeftModel.from_pretrained(
    base_model,
    str(MODEL_DIR)
)

trained_model.eval()

print("\nFinal model loaded for inference.")


# ============================================================
# 11. WORD-LEVEL CANDIDATE GENERATOR
# ============================================================

def get_word_candidates(
    text,
    top_k=5,
    beam_width=12,
    max_new_tokens=8,
):
    """
    Generate candidate continuations and return unique
    first-word candidates.
    """

    inputs = base_tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
    ).to("cuda")

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=beam_width,
            num_return_sequences=beam_width,
            do_sample=False,
            early_stopping=True,
            pad_token_id=base_tokenizer.pad_token_id,
            eos_token_id=base_tokenizer.eos_token_id,
        )

    candidates = []

    for output in outputs:
        generated_ids = output[input_len:]

        text_out = base_tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

        if not text_out:
            continue

        first_word = text_out.split()[0].strip()

        if first_word and first_word not in candidates:
            candidates.append(first_word)

        if len(candidates) >= top_k:
            break

    return candidates


# ============================================================
# 12. QUICK MANUAL TEST
# ============================================================

manual_tests = [
    "aap kya kar rahe",
    "tum kya karte",
    "mujhe kal",
    "aap ne kaha tha",
    "main kal office",
]

print("\n" + "=" * 70)
print("MANUAL TESTS")
print("=" * 70)

for prompt in manual_tests:
    print("\nContext:", prompt)
    print("Candidates:",
          get_word_candidates(prompt, top_k=5))


# ============================================================
# 13. CREATE TEST EXAMPLES
# ============================================================

with TEST_FILE.open("r", encoding="utf-8") as f:
    test_sentences = [
        line.strip()
        for line in f
        if line.strip()
    ]

evaluation_examples = []

for sentence in test_sentences:
    words = sentence.split()

    if len(words) < 2:
        continue

    context = " ".join(words[:-1])
    target = words[-1]

    evaluation_examples.append(
        {
            "context": context,
            "target": target,
        }
    )

print("\nTest examples:", len(evaluation_examples))


# ============================================================
# 14. FULL WORD-LEVEL EVALUATION
# ============================================================

top1 = 0
top3 = 0
top5 = 0

results = []

start_eval = time.time()

for example in tqdm(
    evaluation_examples,
    desc="Evaluating"
):
    context = example["context"]
    target = example["target"]

    candidates = get_word_candidates(
        context,
        top_k=5
    )

    hit1 = (
        len(candidates) >= 1
        and target == candidates[0]
    )

    hit3 = target in candidates[:3]
    hit5 = target in candidates[:5]

    top1 += int(hit1)
    top3 += int(hit3)
    top5 += int(hit5)

    results.append(
        {
            "context": context,
            "target": target,
            "predictions": candidates,
            "top1": hit1,
            "top3": hit3,
            "top5": hit5,
        }
    )

eval_time = time.time() - start_eval

n = len(evaluation_examples)

top1_acc = top1 / n * 100
top3_acc = top3 / n * 100
top5_acc = top5 / n * 100

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Test examples : {n}")
print(f"Top-1 accuracy: {top1_acc:.2f}%")
print(f"Top-3 accuracy: {top3_acc:.2f}%")
print(f"Top-5 accuracy: {top5_acc:.2f}%")
print(f"Evaluation time: {eval_time / 60:.2f} minutes")
print(f"Avg/example: {eval_time / n:.3f} sec")


# ============================================================
# 15. SAVE EVALUATION RESULTS
# ============================================================

results_file = PROJECT_DIR / "evaluation" / "qwen_final_results.json"

results_file.parent.mkdir(parents=True, exist_ok=True)

summary = {
    "model": MODEL_NAME,
    "adapter": str(MODEL_DIR),
    "test_examples": n,
    "top1_accuracy": top1_acc,
    "top3_accuracy": top3_acc,
    "top5_accuracy": top5_acc,
    "evaluation_time_seconds": eval_time,
    "average_seconds_per_example": eval_time / n,
}

with results_file.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "summary": summary,
            "examples": results,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nEvaluation saved to:")
print(results_file)


# ============================================================
# 16. SAVE SIMPLE SUMMARY
# ============================================================

summary_file = PROJECT_DIR / "evaluation" / "final_summary.json"

with summary_file.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to:")
print(summary_file)

print("\nDONE.")

Device: cuda
GPU: Tesla T4

Dataset:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 35137
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 4392
    })
    test: Dataset({
        features: ['text'],
        num_rows: 4393
    })
})


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


Base model loaded.


Adding EOS to train dataset:   0%|          | 0/35137 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/35137 [00:00<?, ? examples/s]

...


Validation results:
{'eval_loss': 4.587658405303955, 'eval_entropy': 4.581867516314218, 'eval_num_tokens': 3540244.0, 'eval_mean_token_accuracy': 0.23518060523709025}
Validation perplexity: 98.26406594176572

GPU memory cleared.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


Final model loaded for inference.

MANUAL TESTS

Context: aap kya kar rahe
Candidates: ['ho', 'hain']

Context: tum kya karte
Candidates: ['ho', 'hai', 'h', 'hain']

Context: mujhe kal
Candidates: ['bhi', '1000000', 'apne']

Context: aap ne kaha tha
Candidates: ['aapne', 'aap', 'apne']

Context: main kal office
Candidates: ['me', 'apne']

Test examples: 4393


Evaluating:   0%|          | 0/4393 [00:00<?, ?it/s]


FINAL TEST RESULTS
Test examples : 4393
Top-1 accuracy: 12.36%
Top-3 accuracy: 19.28%
Top-5 accuracy: 22.81%
Evaluation time: 39.72 minutes
Avg/example: 0.543 sec

Evaluation saved to:
/content/drive/MyDrive/hinglish-next-word-predictor/evaluation/qwen_final_results.json
Summary saved to:
/content/drive/MyDrive/hinglish-next-word-predictor/evaluation/final_summary.json

DONE.


SyntaxError: invalid syntax (2161124335.py, line 3)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Paths
BASE_MODEL = "Qwen/Qwen3-0.6B-Base"
ADAPTER_PATH = "/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora-final"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Load your trained LoRA adapter
trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

trained_model.eval()

print("✅ Tokenizer loaded")
print("✅ Base model loaded")
print("✅ Trained LoRA adapter loaded")
print("Device:", trained_model.device)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora-final'

In [ ]:
import os

base_path = "/content/drive/MyDrive/hinglish-next-word-predictor/models"

print("Models directory exists:", os.path.exists(base_path))

if os.path.exists(base_path):
    print("\nFolders/files inside models:")
    for item in os.listdir(base_path):
        print(" -", item)

target = os.path.join(
    base_path,
    "qwen3-0.6b-hinglish-lora-final"
)

print("\nTarget exists:", os.path.exists(target))

if os.path.exists(target):
    print("\nTarget contents:")
    for item in os.listdir(target):
        print(" -", item)

Models directory exists: False

Target exists: False


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

PROJECT = "/content/drive/MyDrive/hinglish-next-word-predictor"

print("Project exists:", os.path.exists(PROJECT))

if os.path.exists(PROJECT):
    print("\nProject contents:")
    for item in os.listdir(PROJECT):
        print(" -", item)

    models = os.path.join(PROJECT, "models")

    print("\nModels directory exists:", os.path.exists(models))

    if os.path.exists(models):
        print("\nModels contents:")
        for item in os.listdir(models):
            print(" -", item)

Mounted at /content/drive
Project exists: True

Project contents:
 - notebooks
 - data
 - preprocessing
 - training
 - models
 - inference
 - evaluation

Models directory exists: True

Models contents:
 - qwen3-0.6b-hinglish-lora
 - qwen3-0.6b-hinglish-lora-final


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/hinglish-next-word-predictor"

matches = []

for root, dirs, files in os.walk(PROJECT):
    if "adapter_config.json" in files:
        matches.append(root)

print("Adapter folders found:")

if matches:
    for path in matches:
        print(path)
else:
    print("❌ No adapter_config.json found")

Adapter folders found:
/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora
/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora/dry_run
/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora-final
/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora-final/checkpoint-1500
/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora-final/checkpoint-1738


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-0.6B-Base"
ADAPTER_PATH = "/content/drive/MyDrive/hinglish-next-word-predictor/models/qwen3-0.6b-hinglish-lora-final"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

# Load base Qwen model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Load your FULL trained LoRA adapter
trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

trained_model.eval()

print("✅ Tokenizer loaded")
print("✅ Base model loaded")
print("✅ FULL trained LoRA adapter loaded")
print("Device:", trained_model.device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
!pip install -q "torchao>=0.18.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.3 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
import torch
import torchao
import peft
import transformers

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())


PyTorch: 2.11.0+cpu
TorchAO: 0.18.0
PEFT: 0.20.0
Transformers: 5.16.1
CUDA: False
